In [106]:
import pandas as pd
import numpy as np
import zipfile
from datetime import datetime
import os
pd.set_option("display.max_colwidth", None)



In [107]:
espacios_col = [(0, 3), (3, 23), (23, 43), (43,45), (45, 48), (48, 49), (368, 376), (376, 384), (384, 392), (346, 347), (411, 426)
                ]

column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda", 
                "Tipo de Movimiento", "Fecha de Afiliacion", "Fecha de inicio del seguro", "Fecha fin del seguro",
                "Periodo de pago", "Prima"
                ]

In [124]:
def Extraer_fechas(filename):
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        # Primera fecha (ddmmyy → fecha completa)
        date_str = base[3:9]   # "030120"
        fecha_trama = datetime.strptime(date_str, "%d%m%y").date()

        # Segunda fecha (ddmm → usar año de la primera fecha)
        decl_str = base[10:14]  # "0601"
        dia, mes = int(decl_str[:2]), int(decl_str[2:])
        fecha_declarada = datetime(fecha_trama.year, mes, dia).date()

        return fecha_trama, fecha_declarada
    except Exception:
        return None, None  # Si algo falla


In [125]:
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [126]:
# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")       
    return row

In [127]:
archivo_zip = []
with zipfile.ZipFile('C:/data/TRAMAS/RED_2020-01.zip', "r") as archivo_zip:
    tramas_txt = archivo_zip.namelist()
    #for file in tramas_txt:
        #with archivo_zip.open(file) as file_txt:
        #    file_content = file_txt.read()
        #    list_lines = file_content.decode('latin-1').splitlines()
    file_content= archivo_zip.read(tramas_txt[2])
    lista_lineas = file_content.decode('latin-1').splitlines()
    print(lista_lineas[0:3])
print(tramas_txt)


['703001101013140001502280101                00PEN4                                                                                                                                                                                                                                                                                                         M                     202001062019122720200127000{00000000600000{00000000000480I                                                                                                                                                                                                                                                                                          ', '703001101021840002808010102                00PEN4                                                                                                                                                                                                                                             

In [128]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])

In [129]:
letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))
def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan
        
        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


In [130]:
df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
df_tramas["Error Prima"] = df_tramas["Prima Bruta"].isna()

In [131]:
df_tramas["Fecha de Afiliacion"] = Convertir_fecha(df_tramas["Fecha de Afiliacion"])
df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

In [132]:
fecha_trama, fecha_declarada = Extraer_fechas(tramas_txt[2])
df_tramas["Fecha Trama"] = fecha_trama
df_tramas["Fecha Declarada"] = fecha_declarada
df_tramas['Fecha Trama'] = pd.to_datetime(df_tramas['Fecha Trama'], errors='coerce')
df_tramas['Fecha Declarada'] = pd.to_datetime(df_tramas['Fecha Declarada'], errors='coerce')

In [133]:
df_tramas.columns = (df_tramas.columns
                     .str.strip()  # quitar espacios al inicio/fin
                     .str.upper()  # opcional: todo en mayúsculas
                     .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
)

In [134]:
df_tramas.head()

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA,FECHA_TRAMA,FECHA_DECLARADA
0,703,00110101314000150228,0101,00,PEN,4,2020-01-06,2019-12-27,2020-01-27,M,00000000000480I,703001101013140001502280101 00PEN4 M 202001062019122720200127000{00000000600000{00000000000480I,48.09,False,2020-01-06,2020-01-07
1,703,00110102184000280801,0102,00,PEN,4,2020-01-06,2020-01-05,2020-02-05,M,00000000000480I,703001101021840002808010102 00PEN4 M 202001062020010520200205000{00000000600000{00000000000480I,48.09,False,2020-01-06,2020-01-07
2,703,00110102144000280836,0102,00,PEN,4,2020-01-06,2020-01-04,2020-02-04,M,00000000000480I,703001101021440002808360102 00PEN4 M 202001062020010420200204000{00000000600000{00000000000480I,48.09,False,2020-01-06,2020-01-07
3,703,00110102114000281069,0102,00,PEN,4,2020-01-06,2020-01-06,2020-02-06,M,00000000000480I,703001101021140002810690102 00PEN4 M 202001062020010620200206000{00000000600000{00000000000480I,48.09,False,2020-01-06,2020-01-07
4,703,00110108824000131809,0108,00,PEN,2,2020-01-06,2012-03-28,2020-01-29,M,00000000000480I,703001101088240001318090108 00PEN2 M 202001062012032820200129000{00000000600000{00000000000480I,48.09,False,2020-01-06,2020-01-07


In [72]:
df_filtrado = df_tramas[df_tramas["Prima_Bruta"].isna()]
df_filtrado.head(3)

,Tipo de seguro,Certificado,Numero Interno Del Canal,Tipo de Registro,Moneda,Tipo de Movimiento,Fecha de Afiliación,Fecha de inicio del seguro,Fecha fin del seguro,Periodo de pago,Prima,Trama Original,Prima_Bruta
2458,815,00110465264001212138,0345,03,PEN,0,,,,,,815001104652640012121380345 03PEN0000398 L20897145 PEN000000000054I20200106,NaN
2460,815,00110465294001364248,0465,03,PEN,0,,,,,,815001104652940013642480465 03PEN0000686 L08173036 PEN000000000054I20200106,NaN
2462,815,00110465244001418526,0465,03,PEN,0,,,,,,815001104652440014185260465 03PEN0000829 L08550585 PEN000000000054I20200106,NaN


In [135]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149593 entries, 0 to 149592
Data columns (total 16 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   TIPO_DE_SEGURO              149593 non-null  object        
 1   CERTIFICADO                 149593 non-null  object        
 2   NUMERO_INTERNO_DEL_CANAL    149593 non-null  object        
 3   TIPO_DE_REGISTRO            149593 non-null  object        
 4   MONEDA                      149593 non-null  object        
 5   TIPO_DE_MOVIMIENTO          149593 non-null  object        
 6   FECHA_DE_AFILIACION         91156 non-null   datetime64[ns]
 7   FECHA_DE_INICIO_DEL_SEGURO  99103 non-null   datetime64[ns]
 8   FECHA_FIN_DEL_SEGURO        99449 non-null   datetime64[ns]
 9   PERIODO_DE_PAGO             149593 non-null  object        
 10  PRIMA                       149593 non-null  object        
 11  TRAMA_ORIGINAL              149593 non-